In [105]:
import scanpy as sc

In [65]:
import os
os.chdir("/home/lishaogang/NEU2026_May/GEO_data/")

In [66]:
import os
import scanpy as sc
import anndata as ad
import pandas as pd

# 1. 设置文件夹路径
data_dir = '/home/lishaogang/NEU2026_May/GEO_data'

# 2. 【核心修改】精确指定你需要的“白名单”文件列表
# 你可以在这里随意注释掉（加 #）或者删除你不需要的样本
target_files = [
    "GSE133028_raw_count.h5ad",
    "GSE134577_raw_count.h5ad",
    "GSE138266_raw_count_rmlowquality.h5ad",
    "GSE141578_raw_count.h5ad",
    "GSE163005_raw_count.h5ad",
    "GSE186802_raw_count_subset.h5ad",
    "GSE194078_raw_count.h5ad",
    "GSE200164_raw_count.h5ad",
    "GSE243905_raw_count.h5ad",
    "GSE244499_raw_count.h5ad",
    # "GSE247134_raw_count.h5ad",  # 假设你不要这个，直接注释掉即可
    "GSE247314_raw_count.h5ad" 
]

adata_dict = {}
stats_list = []

print(f"🚀 启动精准白名单模式，准备读取 {len(target_files)} 个指定文件...\n")

# 3. 按名单逐个点名读取
for filename in target_files:
    file_path = os.path.join(data_dir, filename)
    dataset_name = filename.replace('.h5ad', '')
    
    # 检查文件是否存在，防止拼写错误报错
    if os.path.exists(file_path):
        try:
            adata = sc.read_h5ad(file_path)
            adata_dict[dataset_name] = adata
            
            stats_list.append({
                'Dataset': dataset_name,
                'Cells': adata.n_obs,
                'Genes': adata.n_vars
            })
            print(f"✅ 成功载入: {dataset_name:<35} | 细胞数: {adata.n_obs}")
        except Exception as e:
            print(f"❌ 载入失败 {filename}: {e}")
    else:
        print(f"⚠️ 找不到文件，已跳过: {filename}")

# 打印一下当前的统计全貌
df_stats = pd.DataFrame(stats_list)
print(f"\n📊 成功读取 {len(adata_dict)} 个数据集，总细胞数: {df_stats['Cells'].sum():,}！")



In [77]:
# 遍历所有数据集，打印每个 adata.obs 的前两行
print("="*80)
print("📄 查看每个数据集 adata.obs 前两行信息\n")

for name, adata in adata_dict.items():
    print(f"🔹 Dataset: {name}")
    print(f"细胞数: {adata.n_obs}, 基因数: {adata.n_vars}")
    # 显示前两行，所有列
    print(adata.obs.head(2))
    print("-"*80 + "\n")

In [70]:
# 逐个修复 orig.ident
adata_dict['GSE134577_raw_count'].obs['orig.ident'] = adata_dict['GSE134577_raw_count'].obs['GSM'].copy()

adata_dict['GSE186802_raw_count_subset'].obs['orig.ident'] = adata_dict['GSE186802_raw_count_subset'].obs['GSM_ID_1'].copy()

adata_dict['GSE194078_raw_count'].obs['orig.ident'] = adata_dict['GSE194078_raw_count'].obs['GSM_ID'].copy()

adata_dict['GSE243905_raw_count'].obs['orig.ident'] = adata_dict['GSE243905_raw_count'].obs['GSM_ID'].copy()


In [79]:
247314

In [84]:
adata_dict['GSE247314_raw_count'].obs['orig.ident'] = adata_dict['GSE247314_raw_count'].obs['GSM_ID'].copy()

In [72]:
import os
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd
from upsetplot import from_contents, plot

print(">>> 启动基因集提纯与 UpSet 图绘制流水线...\n")

# 1. 过滤“幽灵基因”并构建基因集合字典
gene_contents = {}
print("正在逐个排查数据集，剔除全为 0 的未表达基因：")

for dataset_key, adata in adata_dict.items():
    gse_id = dataset_key.split('_')[0]
    
    # 记录过滤前的基因数，用于对比
    n_genes_before = adata.n_vars
    
    # 【核心修改点】：过滤掉在该数据集中所有细胞都不表达（表达量为 0）的基因
    # min_cells=1 表示：一个基因至少要在 1 个细胞中有表达才会被保留
    sc.pp.filter_genes(adata, min_cells=1)
    
    n_genes_after = adata.n_vars
    dropped_genes = n_genes_before - n_genes_after
    
    # 提取过滤后的真实表达基因名，转为 set 以大幅提升求交集的速度
    gene_contents[gse_id] = set(adata.var_names)
    
    print(f"  ✅ {gse_id:<12} | 移除了 {dropped_genes:>5} 个未表达基因 | 剩余有效基因: {n_genes_after}")

# 2. 将字典数据转换为 UpSet 专属的 DataFrame
print("\n>>> 正在计算提纯后的交集矩阵并生成 UpSet 图...")
upset_data = from_contents(gene_contents)

# 3. 启动绘图引擎
fig = plt.figure(figsize=(18, 10))

# 核心绘制函数
plot(
    upset_data,
    fig=fig,
    orientation='horizontal',   # 柱状图横向排列
    sort_by='cardinality',      # 按照交集包含的基因数量从大到小排序
    min_subset_size=100,        # 只显示交集基因数 >= 100 个的组合
    show_counts=True            # 标注具体的基因个数
)

# 4. 润色标题与布局
plt.suptitle("Expressed Gene Intersections Across Multi-Center Datasets", fontsize=16, y=0.95)

# 5. 保存图表
output_fig_path = '/home/lishaogang/NEU2026_May/GEO_data/datasets_expressed_genes_upset_plot.png'
plt.savefig(output_fig_path, bbox_inches='tight', dpi=300)
plt.show()

print(f"\n🎉 绘制完成！高清提纯版 UpSet 图已成功保存至服务器: {output_fig_path}")

In [85]:
import anndata as ad

print(">>> 🔍 正在全网通缉丢失 'orig.ident' 列的数据集...")

# 1. 遍历字典，揪出缺失列的“内鬼”并当场修复
for name, adata in adata_dict.items():
    if 'orig.ident' not in adata.obs.columns:
        print(f"  🚨 抓到缺漏！[{name}] 缺少 orig.ident")
        
        # 补救策略：如果它有 GSM_ID，就用 GSM_ID 填补；如果没有，用当前数据集名字填补
        if 'GSM_ID' in adata.obs.columns:
            adata.obs['orig.ident'] = adata.obs['GSM_ID']
            print(f"     ✅ 已使用 GSM_ID 自动修补！")
        else:
            adata.obs['orig.ident'] = name
            print(f"     ✅ 已使用 Dataset Name 自动修补！")

print("\n>>> 🚀 列对齐完毕，重新启动百万级合并引擎...")

# 2. 再次执行核心大合并
# index_unique='-' 会在合并时直接利用源数据集的标签后缀处理极少数的重名细胞
adata_mega = ad.concat(adata_dict, join='inner', label='dataset', index_unique='-')
adata_mega.obs_names_make_unique()

# 3. 极致瘦身
if 'orig.ident' in adata_mega.obs.columns:
    adata_mega.obs = adata_mega.obs[['orig.ident', 'dataset']].copy()
else:
    print("⚠️ 仍然找不到 orig.ident，请检查上述修复逻辑。")

# 4. 打印最终形态
print(f"\n🎉 抢救大成功！")
print(f"👉 当前超级矩阵形态: {adata_mega.n_obs} 个细胞, {adata_mega.n_vars} 个核心共享基因")
print("\n👇 查看抢救后的 obs 矩阵头部预览：")
print(adata_mega.obs.head())

In [87]:


# ==========================================
# 4. (可选) 保存这个极其纯净的超级对象
# ==========================================
out_path = '/home/lishaogang/NEU2026_May/GEO_data/MEGA_PAN_DISEASE_CLEAN.h5ad'
adata_mega.write(out_path)


In [86]:

# 标记以GSM开头的样本，na=False把缺失值判定为False
is_gsm_mask = adata_mega.obs['orig.ident'].str.startswith('GSM', na=False)

# 全局判断是否全部符合
all_match = is_gsm_mask.all()
print("全部orig.ident以GSM开头：", all_match)

# 提取异常行
abnormal_df = adata_mega.obs[~is_gsm_mask]
print(f"\n异常细胞数量：{len(abnormal_df)}")

if len(abnormal_df) > 0:
    # 打印所有非GSM的样本ID
    bad_sample_ids = abnormal_df['orig.ident'].unique()
    print(f"\n异常样本ID列表（共{len(bad_sample_ids)}种）：")
    print(bad_sample_ids)
    
    # 统计每个dataset里的异常样本分布
    print("\n=== 各数据集异常样本明细 ===")
    group_stat = abnormal_df.groupby(['dataset', 'orig.ident']).size().reset_index(name='cell_num')
    print(group_stat)
else:
    print("\n✅ 所有orig.ident都是GSM开头，无异常")


In [88]:
pb_50_genes = [
    # 核糖体蛋白 (RPS/RPL 系列)
    'RPS3', 'RPS4X', 'RPS6', 'RPS8', 'RPS10', 'RPS12', 'RPS13', 'RPS14', 'RPS18', 'RPS19', 
    'RPL3', 'RPL4', 'RPL5', 'RPL7', 'RPL8', 'RPL9', 'RPL10', 'RPL13', 'RPL18', 'RPL21', 
    'RPL23A', 'RPL27', 'RPL28', 'RPL30', 'RPL31', 'RPL32', 'RPL34', 'RPL35', 'RPL37', 'RPL38',
    # 细胞骨架与代谢核心
    'ACTB', 'GAPDH', 'TUBA1B', 'TMSB10', 'TMSB4X', 'PTPRC', 'EEF1A1', 'H3C1', 'H3C2', 'UBB',
    # 免疫通用标记
    'B2M', 'HLA-A', 'HLA-B', 'HLA-C', 'CD52', 'CTSS', 'CD74', 'LYZ', 'HLA-DRA', 'HLA-DRB1'
]

In [89]:
import scanpy as sc

# 1. 确定哪些基因在当前的 mega 矩阵中确实存在
# 避免因为基因名不匹配（如大小写或版本差异）导致代码报错
existing_genes = [g for g in pb_50_genes if g in adata_mega.var_names]
missing_genes = set(pb_50_genes) - set(existing_genes)

if missing_genes:
    print(f"⚠️ 警告：以下 {len(missing_genes)} 个基因在当前矩阵中未找到，将跳过: {missing_genes}")

print(f">>> 正在从百万级矩阵中提取 {len(existing_genes)} 个核心基因...")

# 2. 生成 adata_mega_50gene
# 使用 .copy() 确保这是一个独立的新对象，不会影响 adata_mega
adata_mega_50gene = adata_mega[:, existing_genes].copy()

# 3. 验证结果
print(f"\n✅ 提取成功！")
print(f"   对象形态: {adata_mega_50gene.n_obs} 个细胞, {adata_mega_50gene.n_vars} 个核心基因")

# 4. 快速检查一下这些基因在所有细胞中的表达情况 (可选)
# 这能让你一眼看出这些管家基因在你的数据中是否表达稳定
sc.pl.dotplot(adata_mega_50gene, var_names=adata_mega_50gene.var_names, groupby='dataset', 
              save='_50_genes_check.pdf')
print("\n🎉 50 基因矩阵已生成，并已保存管家基因表达 DotPlot 供质检。")

In [90]:
adata_old = sc.read_h5ad("/home/lishaogang/NEU2026-Jan/adata_final.h5ad")
# 1. 使用我们在 adata_mega_50gene 中确认过的 existing_genes 列表
# 确保两者的维度完全一致，方便后续进行直接对比
adata_old_50gene = adata_old[:, existing_genes].copy()

# 2. 验证结果
print(f"✅ 提取成功！")
print(f"   旧版 50 基因矩阵形态: {adata_old_50gene.n_obs} 个细胞, {adata_old_50gene.n_vars} 个基因")

# 3. 简单的质检：对比一下新旧版在这 48 个基因上表达量的大致分布
print("\n🔍 简要统计对比:")
print(f"   新版 (adata_mega) 50 基因均值: {adata_mega_50gene.X.mean():.4f}")
print(f"   旧版 (adata_old) 50 基因均值: {adata_old_50gene.X.mean():.4f}")

In [92]:
import numpy as np
import pandas as pd

# =========================================================
# 检查 adata_old_50gene 的 orig.ident 是否都存在于 adata_mega_50gene
# =========================================================

old_idents = pd.Series(adata_old_50gene.obs["orig.ident"].astype(str).unique())
mega_idents = pd.Series(adata_mega_50gene.obs["orig.ident"].astype(str).unique())

old_set = set(old_idents)
mega_set = set(mega_idents)

missing_in_mega = sorted(old_set - mega_set)
extra_in_mega = sorted(mega_set - old_set)
common_idents = sorted(old_set & mega_set)

print(">>> adata_old_50gene 中 orig.ident 数量:", len(old_set))
print(">>> adata_mega_50gene 中 orig.ident 数量:", len(mega_set))
print(">>> 两者共有 orig.ident 数量:", len(common_idents))

print("\n>>> old 中有，但 mega 中没有的 orig.ident 数量:", len(missing_in_mega))
print(missing_in_mega)

print("\n>>> mega 中有，但 old 中没有的 orig.ident 数量:", len(extra_in_mega))
print(extra_in_mega)

if len(missing_in_mega) == 0:
    print("\n✅ adata_old_50gene 里的所有 orig.ident 都存在于 adata_mega_50gene 中")
else:
    print("\n⚠️ adata_old_50gene 中有部分 orig.ident 不存在于 adata_mega_50gene 中")

In [94]:
import numpy as np
import pandas as pd
from scipy import sparse

print(">>> 🚀 启动基于 48/50 个基因表达相关性的跨矩阵细胞映射...")
print(">>> mega 使用全基因 raw count 计算 CP10K，再提取目标基因")


# =========================================================
# 0. 基础检查
# =========================================================

assert "cp10k_counts" in adata_old_50gene.layers.keys(), \
    "adata_old_50gene.layers 中没有 cp10k_counts"

assert "orig.ident" in adata_old_50gene.obs.columns, \
    "adata_old_50gene.obs 中没有 orig.ident"

assert "orig.ident" in adata_mega.obs.columns, \
    "adata_mega.obs 中没有 orig.ident"

assert adata_old_50gene.obs_names.is_unique, \
    "adata_old_50gene.obs_names 不唯一"

assert adata_mega.obs_names.is_unique, \
    "adata_mega.obs_names 不唯一"


# =========================================================
# 1. 检查并对齐目标基因
# =========================================================

target_genes = adata_old_50gene.var_names.astype(str)

common_genes = target_genes.intersection(adata_mega.var_names.astype(str))

print(f">>> old 目标基因数: {adata_old_50gene.n_vars}")
print(f">>> adata_mega 全基因数: {adata_mega.n_vars}")
print(f">>> old 与 mega 共同目标基因数: {len(common_genes)}")

missing_genes = target_genes.difference(adata_mega.var_names.astype(str))

if len(missing_genes) > 0:
    print(">>> ⚠️ mega 中缺失以下目标基因:")
    print(list(missing_genes))

assert len(common_genes) == adata_old_50gene.n_vars, \
    "adata_mega 中缺少 adata_old_50gene 的部分目标基因，不能继续匹配"

# 按 old 的基因顺序，从 mega 中取目标基因
adata_mega_gene = adata_mega[:, target_genes].copy()

assert np.all(adata_mega_gene.var_names == adata_old_50gene.var_names), \
    "old 和 mega 目标基因顺序不一致"

print(">>> ✅ 目标基因顺序检查通过")


# =========================================================
# 2. 用 adata_mega 全基因 raw count 计算每个细胞 total_counts
# =========================================================

print(">>> 正在计算 adata_mega 全基因 total_counts...")

if sparse.issparse(adata_mega.X):
    mega_total_counts = np.asarray(adata_mega.X.sum(axis=1)).ravel()
else:
    mega_total_counts = np.asarray(adata_mega.X.sum(axis=1)).ravel()

mega_total_counts = mega_total_counts.astype(np.float32)

print(">>> mega_total_counts 描述:")
print(pd.Series(mega_total_counts).describe())

n_zero_total = np.sum(mega_total_counts == 0)
print(f">>> total_counts 为 0 的 mega 细胞数: {n_zero_total}")

if n_zero_total > 0:
    print(">>> ⚠️ 有些 mega 细胞全基因 count 为 0，这些细胞后续 CP10K 会被设为 0")


# =========================================================
# 3. 工具函数
# =========================================================

def get_dense_matrix_from_X_or_layer(adata, layer=None):
    """
    提取 AnnData 的 X 或 layer，返回 dense float32 numpy array。
    """
    if layer is None:
        mat = adata.X
    else:
        mat = adata.layers[layer]
    
    if sparse.issparse(mat):
        mat = mat.toarray()
    else:
        mat = np.asarray(mat)
    
    return mat.astype(np.float32)


def get_mega_cp10k_for_cells(cells_mega_gene, total_counts_for_cells):
    """
    对 mega 的目标基因 raw count 做 CP10K。
    注意 denominator 是全基因 total_counts，不是目标基因 total_counts。
    """
    mat = get_dense_matrix_from_X_or_layer(cells_mega_gene, layer=None)
    
    denom = total_counts_for_cells.astype(np.float32).reshape(-1, 1)
    
    # 避免除以 0
    denom_safe = np.where(denom > 0, denom, np.inf)
    
    mat_cp10k = mat / denom_safe * 1e4
    
    # total_counts 为 0 的细胞设为 0
    mat_cp10k[~np.isfinite(mat_cp10k)] = 0
    
    return mat_cp10k.astype(np.float32)


def row_zscore(mat, eps=1e-9):
    """
    对每个细胞的目标基因表达向量做 row-wise z-score。
    Pearson 相关系数后续用 dot / n_genes。
    """
    mean = mat.mean(axis=1, keepdims=True)
    std = mat.std(axis=1, keepdims=True)
    
    mat_z = (mat - mean) / (std + eps)
    
    # 防止出现 nan / inf
    mat_z[~np.isfinite(mat_z)] = 0
    
    return mat_z.astype(np.float32)


# =========================================================
# 4. 统一 orig.ident 类型，避免 str / float / NaN 混合报错
# =========================================================

adata_old_50gene.obs["_orig_ident_str"] = adata_old_50gene.obs["orig.ident"].astype("object")
adata_mega_gene.obs["_orig_ident_str"] = adata_mega_gene.obs["orig.ident"].astype("object")

old_valid_ident = adata_old_50gene.obs["_orig_ident_str"].notna()
mega_valid_ident = adata_mega_gene.obs["_orig_ident_str"].notna()

adata_old_50gene.obs.loc[old_valid_ident, "_orig_ident_str"] = (
    adata_old_50gene.obs.loc[old_valid_ident, "_orig_ident_str"].astype(str)
)

adata_mega_gene.obs.loc[mega_valid_ident, "_orig_ident_str"] = (
    adata_mega_gene.obs.loc[mega_valid_ident, "_orig_ident_str"].astype(str)
)

old_idents = adata_old_50gene.obs.loc[old_valid_ident, "_orig_ident_str"].unique()
mega_idents = adata_mega_gene.obs.loc[mega_valid_ident, "_orig_ident_str"].unique()

common_idents = sorted(set(old_idents) & set(mega_idents))
missing_in_mega = sorted(set(old_idents) - set(mega_idents))
extra_in_mega = sorted(set(mega_idents) - set(old_idents))

print(f">>> old orig.ident 数量: {len(old_idents)}")
print(f">>> mega orig.ident 数量: {len(mega_idents)}")
print(f">>> 共有 orig.ident 数量: {len(common_idents)}")
print(f">>> old 中有但 mega 中没有的 orig.ident 数量: {len(missing_in_mega)}")

if len(missing_in_mega) > 0:
    print(">>> old 中有但 mega 中没有的 orig.ident:")
    print(missing_in_mega)


# =========================================================
# 5. 准备结果列
# =========================================================

adata_old_50gene.obs["raw_cell_id"] = pd.Series(
    index=adata_old_50gene.obs_names,
    dtype="object"
)

adata_old_50gene.obs["raw_cell_corr"] = np.nan


# =========================================================
# 6. 按 orig.ident 内部进行分块相关性匹配
# =========================================================

chunk_size = 1000

print("\n>>> 开始逐 orig.ident 匹配...")

for ident in common_idents:
    print("\n" + "=" * 100)
    print(f">>> 正在处理样本: {ident}")
    
    old_mask = adata_old_50gene.obs["_orig_ident_str"] == ident
    mega_mask = adata_mega_gene.obs["_orig_ident_str"] == ident
    
    old_idx = np.where(old_mask.to_numpy())[0]
    mega_idx = np.where(mega_mask.to_numpy())[0]
    
    cells_old = adata_old_50gene[old_idx, :].copy()
    cells_mega_gene = adata_mega_gene[mega_idx, :].copy()
    
    n_old = cells_old.n_obs
    n_mega = cells_mega_gene.n_obs
    
    print(f"    old cells : {n_old}")
    print(f"    mega cells: {n_mega}")
    
    if n_old == 0 or n_mega == 0:
        print(f"    ⚠️ 跳过 {ident}，因为 old 或 mega 细胞数为 0")
        continue
    
    # old 使用已有的 cp10k_counts
    mat_old_cp10k = get_dense_matrix_from_X_or_layer(
        cells_old,
        layer="cp10k_counts"
    )
    
    # mega 使用全基因 total_counts 计算目标基因 CP10K
    total_counts_for_mega_cells = mega_total_counts[mega_idx]
    
    mat_mega_cp10k = get_mega_cp10k_for_cells(
        cells_mega_gene,
        total_counts_for_mega_cells
    )
    
    # 检查目标基因全 0 的细胞
    old_target_sum = mat_old_cp10k.sum(axis=1)
    mega_target_sum = mat_mega_cp10k.sum(axis=1)
    
    n_old_zero_target = np.sum(old_target_sum == 0)
    n_mega_zero_target = np.sum(mega_target_sum == 0)
    
    print(f"    old 目标基因全 0 细胞数 : {n_old_zero_target}")
    print(f"    mega 目标基因全 0 细胞数: {n_mega_zero_target}")
    
    mat_old_z = row_zscore(mat_old_cp10k)
    mat_mega_z = row_zscore(mat_mega_cp10k)
    
    n_genes = mat_old_z.shape[1]
    
    best_names_all = []
    best_corr_all = []
    
    for start in range(0, n_old, chunk_size):
        end = min(start + chunk_size, n_old)
        
        old_block_z = mat_old_z[start:end, :]
        
        # Pearson correlation
        # row_zscore 使用 ddof=0，所以这里除以 n_genes
        corr_block = np.dot(old_block_z, mat_mega_z.T) / n_genes
        
        best_idx = np.argmax(corr_block, axis=1)
        best_corr = corr_block[np.arange(corr_block.shape[0]), best_idx]
        best_names = cells_mega_gene.obs_names[best_idx].to_numpy()
        
        best_names_all.extend(best_names)
        best_corr_all.extend(best_corr)
        
        print(f"    processed old cells {start} - {end}")
    
    adata_old_50gene.obs.loc[cells_old.obs_names, "raw_cell_id"] = best_names_all
    adata_old_50gene.obs.loc[cells_old.obs_names, "raw_cell_corr"] = best_corr_all
    
    print(f"    ✅ 样本 {ident} 映射完成")


# =========================================================
# 7. 映射结果概览
# =========================================================

n_mapped = adata_old_50gene.obs["raw_cell_id"].notna().sum()
n_total = adata_old_50gene.n_obs

print("\n" + "=" * 100)
print(">>> ✅ 全部映射完成")
print(f">>> 成功映射细胞数: {n_mapped} / {n_total}")
print(f">>> 映射比例: {n_mapped / n_total:.2%}")

print("\n>>> raw_cell_corr 描述统计:")
print(adata_old_50gene.obs["raw_cell_corr"].describe())

dup_rate = adata_old_50gene.obs.loc[
    adata_old_50gene.obs["raw_cell_id"].notna(),
    "raw_cell_id"
].duplicated().mean()

print(f"\n>>> raw_cell_id 重复匹配比例: {dup_rate:.2%}")


# =========================================================
# 8. 按 orig.ident 查看匹配质量
# =========================================================

corr_by_ident = (
    adata_old_50gene.obs
    .dropna(subset=["raw_cell_corr"])
    .groupby("_orig_ident_str")["raw_cell_corr"]
    .describe()
    .sort_values("mean")
)

print("\n>>> 按 orig.ident 的 raw_cell_corr 统计:")
print(corr_by_ident.to_string())


# =========================================================
# 9. 随机抽 10 对细胞，查看目标基因表达值
# =========================================================

print("\n>>> 📊 随机抽样验证 10 对细胞...")

mapped_obs = adata_old_50gene.obs[
    adata_old_50gene.obs["raw_cell_id"].notna()
].copy()

n_sample = min(10, mapped_obs.shape[0])

sample_old_names = np.random.choice(
    mapped_obs.index.to_numpy(),
    size=n_sample,
    replace=False
)

for old_name in sample_old_names:
    mega_name = adata_old_50gene.obs.loc[old_name, "raw_cell_id"]
    corr_value = adata_old_50gene.obs.loc[old_name, "raw_cell_corr"]
    
    # old vector
    old_vec = get_dense_matrix_from_X_or_layer(
        adata_old_50gene[old_name, :],
        layer="cp10k_counts"
    ).flatten()
    
    # mega vector：这里仍然用全基因 total_counts 计算 CP10K
    mega_pos = adata_mega_gene.obs_names.get_loc(mega_name)
    
    mega_raw_vec = get_dense_matrix_from_X_or_layer(
        adata_mega_gene[mega_name, :],
        layer=None
    ).flatten()
    
    mega_total = mega_total_counts[mega_pos]
    
    if mega_total > 0:
        mega_vec = mega_raw_vec / mega_total * 1e4
    else:
        mega_vec = np.zeros_like(mega_raw_vec, dtype=np.float32)
    
    check_df = pd.DataFrame({
        "gene": adata_old_50gene.var_names,
        "old_cp10k": old_vec,
        "mega_cp10k": mega_vec,
        "abs_diff": np.abs(old_vec - mega_vec)
    })
    
    print("\n" + "=" * 100)
    print(f"old cell : {old_name}")
    print(f"mega cell: {mega_name}")
    print(f"orig.ident: {adata_old_50gene.obs.loc[old_name, 'orig.ident']}")
    print(f"Pearson corr: {corr_value:.4f}")
    print(check_df.head(10).to_string(index=False))

In [96]:
#将adata_old_50gene中新增的raw_cell_id的值映射回adata_old
import scanpy as sc

# 1. 提取映射表：细胞索引 → raw_cell_id
map_df = adata_old_50gene.obs[["raw_cell_id"]].copy()

# 2. 将映射表转为字典（索引为key，raw_cell_id为value）
cell2rawid = map_df["raw_cell_id"].to_dict()

# 3. 映射回 adata_old 的 obs
# 不在 adata_old_50gene 里的细胞填充 NaN
adata_old.obs["raw_cell_id"] = adata_old.obs.index.map(cell2rawid)

# 4. 校验结果
print("映射完成，前10行查看：")
print(adata_old.obs[["raw_cell_id"]].head(10))

# 统计缺失值（不在子集里的细胞）
print(f"\n无匹配raw_cell_id的细胞数量：{adata_old.obs['raw_cell_id'].isna().sum()}")

In [97]:
import numpy as np
import pandas as pd

# =========================================================
# 1. 保存原始细胞名
# =========================================================

adata_old.obs["old_cell_id"] = adata_old.obs_names.astype(str)

# =========================================================
# 2. 检查 raw_cell_id 是否缺失和唯一
# =========================================================

print("raw_cell_id 缺失数量:", adata_old.obs["raw_cell_id"].isna().sum())
print("adata_old 细胞数:", adata_old.n_obs)
print("raw_cell_id 唯一数量:", adata_old.obs["raw_cell_id"].nunique())

dup_raw_ids = adata_old.obs["raw_cell_id"][
    adata_old.obs["raw_cell_id"].duplicated()
]

print("重复 raw_cell_id 数量:", dup_raw_ids.shape[0])

if dup_raw_ids.shape[0] > 0:
    print("重复 raw_cell_id 示例:")
    print(dup_raw_ids.head(20))

In [98]:
# 将 raw_cell_corr 也映射回 adata_old
cell2corr = adata_old_50gene.obs["raw_cell_corr"].to_dict()

adata_old.obs["raw_cell_corr"] = adata_old.obs.index.map(cell2corr)

print("raw_cell_corr 缺失数量:", adata_old.obs["raw_cell_corr"].isna().sum())
print(adata_old.obs[["raw_cell_id", "raw_cell_corr"]].head())

In [99]:
# 将 raw_cell_corr 也映射回 adata_old
cell2corr = adata_old_50gene.obs["raw_cell_corr"].to_dict()

adata_old.obs["raw_cell_corr"] = adata_old.obs.index.map(cell2corr)

print("raw_cell_corr 缺失数量:", adata_old.obs["raw_cell_corr"].isna().sum())
print(adata_old.obs[["raw_cell_id", "raw_cell_corr"]].head())

In [100]:
# =========================================================
# 每个 raw_cell_id 只保留 raw_cell_corr 最高的 old cell
# =========================================================

adata_old.obs["old_cell_id"] = adata_old.obs_names.astype(str)

obs_tmp = adata_old.obs.copy()

# 按 raw_cell_id 分组，保留 raw_cell_corr 最大的那一行
# 如果相关系数完全一样，则保留第一行
keep_old_cells = (
    obs_tmp
    .sort_values("raw_cell_corr", ascending=False)
    .drop_duplicates(subset="raw_cell_id", keep="first")
    .index
)

print("原始细胞数:", adata_old.n_obs)
print("保留细胞数:", len(keep_old_cells))
print("删除重复细胞数:", adata_old.n_obs - len(keep_old_cells))

adata_old_unique = adata_old[keep_old_cells].copy()

# 恢复原始顺序，可选
adata_old_unique = adata_old_unique[
    obs_tmp.loc[keep_old_cells].sort_index().index
].copy()

print(adata_old_unique)

In [102]:
print("raw_cell_id 缺失数量:", adata_old_unique.obs["raw_cell_id"].isna().sum())
print("adata_old_unique 细胞数:", adata_old_unique.n_obs)
print("raw_cell_id 唯一数量:", adata_old_unique.obs["raw_cell_id"].nunique())
print("raw_cell_id 是否唯一:", adata_old_unique.obs["raw_cell_id"].is_unique)

In [103]:
adata_old_unique.write_h5ad("adata_old_unique.h5ad")

In [104]:
dir()

In [ ]:
del 'adata',
 'adata_GSE163005',
 'adata_GSE163005_common',
 'adata_dict',
 'adata_mega',
 'adata_mega_50gene',
 'adata_mega_common',
 'adata_mega_cp10k',
 'adata_mega_gene',
 'adata_mega_plus_GSE163005',

In [ ]:
import scanpy as sc

In [106]:
adata_old_unique = sc.read_h5ad("adata_old_unique.h5ad")

In [107]:
adata_mega = sc.read_h5ad("MEGA_PAN_DISEASE_CLEAN.h5ad")

In [108]:
assert adata_old_unique.obs["raw_cell_id"].isna().sum() == 0
assert adata_old_unique.obs["raw_cell_id"].is_unique

# 将细胞名改为 raw_cell_id
adata_old_unique.obs_names = adata_old_unique.obs["raw_cell_id"].astype(str).to_numpy()

# 保证 obs 里的 raw_cell_id 和 obs_names 一致
adata_old_unique.obs["raw_cell_id"] = adata_old_unique.obs_names.astype(str)

print(">>> ✅ adata_old_unique 的细胞名已替换为 raw_cell_id")
print(adata_old_unique)

print("\n前 10 个新细胞名:")
print(adata_old_unique.obs_names[:10])

print("\n检查 old_cell_id 和 raw_cell_id:")
print(adata_old_unique.obs[["old_cell_id", "raw_cell_id", "raw_cell_corr"]].head())

In [109]:
common_cells = adata_old_unique.obs_names.intersection(adata_mega.obs_names)

print("adata_old_unique 细胞数:", adata_old_unique.n_obs)
print("adata_mega 细胞数:", adata_mega.n_obs)
print("共同细胞数:", len(common_cells))

missing_in_mega = adata_old_unique.obs_names.difference(adata_mega.obs_names)

print("old_unique 中有但 mega 中没有的细胞数:", len(missing_in_mega))

if len(missing_in_mega) == 0:
    print(">>> ✅ adata_old_unique 的所有细胞名都存在于 adata_mega 中")
else:
    print(">>> ⚠️ 有细胞名不在 adata_mega 中")
    print(missing_in_mega[:20])

In [110]:
adata_old_unique

In [113]:
import numpy as np
from scipy import sparse

# =========================================================
# 1. 取共同基因，并按照 adata_mega 的基因顺序或 old 的基因顺序排列
# =========================================================

common_genes = adata_old_unique.var_names.intersection(adata_mega.var_names)

print("adata_old_unique 基因数:", adata_old_unique.n_vars)
print("adata_mega 基因数:", adata_mega.n_vars)
print("共同基因数:", len(common_genes))

# 建议按照 adata_old_unique 原来的顺序排列
common_genes_ordered = [
    g for g in adata_old_unique.var_names
    if g in common_genes
]

print("共同基因数 ordered:", len(common_genes_ordered))


# =========================================================
# 2. old 和 mega 都按相同细胞、相同基因顺序 subset
# =========================================================

adata_old_common = adata_old_unique[:, common_genes_ordered].copy()

adata_mega_matched = adata_mega[
    adata_old_common.obs_names,
    adata_old_common.var_names
].copy()

# 检查顺序完全一致
assert np.all(adata_old_common.obs_names == adata_mega_matched.obs_names)
assert np.all(adata_old_common.var_names == adata_mega_matched.var_names)

print(">>> ✅ 细胞和基因顺序完全一致")
print("adata_old_common:", adata_old_common)
print("adata_mega_matched:", adata_mega_matched)


# =========================================================
# 3. 将 adata_mega.X 写入 adata_old_common.layers["counts"]
# =========================================================

counts_mat = adata_mega_matched.X

if sparse.issparse(counts_mat):
    counts_mat = counts_mat.tocsr()
else:
    counts_mat = sparse.csr_matrix(counts_mat)

adata_old_common.layers["counts"] = counts_mat

print(">>> ✅ 已添加 counts layer")
print(adata_old_common)
print("layers:", adata_old_common.layers.keys())

In [116]:
x = adata_old_common.layers["counts"]

if sparse.issparse(x):
    values = x.data
else:
    values = np.asarray(x).ravel()
    values = values[values != 0]

integer_like = np.isclose(values, np.round(values), atol=1e-6)

print(">>> counts layer 数值检查:")
print("dtype:", values.dtype)
print("n_nonzero:", len(values))
print("min:", values.min())
print("max:", values.max())
print("接近整数比例:", integer_like.mean())
print("小数比例:", (~integer_like).mean())
print("quantiles:", np.quantile(values, [0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1]))

In [117]:
adata_old_common.write_h5ad("adata_final_0609.h5ad")